## Databricks Homework
Because we are using Databricks Community Edition we will use Databricks File System (DBFS) storage. DBFS is a distributed file system mounted into a Databricks workspace and available on Databricks clusters. In other words you can access DBFS (as well as tables in the schemas) only when cluster is working.  

You could create cluster by selecting 2nd command (which is starting with # dbutils - special utility for databricks ....) and executing following cell by pressing Ctrl + Enter. Window with Attach a Cluster will pop up. Enter cluster name and select Runtime Runtime: 11.3 LTS (Scala 2.12, Spark 3.3.0) (should be by default) then Click Create, Attach, & Run. Wait few minutes until cluster will be created and started. Cluster status will be shown in the top right part. *After some time cluster will shut down if it will be inactive. Due to limitation of community edition each time cluster needs to be recreated if it was shut down. It's normal behaviour.

Please, create table in the default schema using file Sales_December_2019.csv. On the left found Data => Create Table => Drop files to upload, or click to browse => Sales_December_2019.csv After file will be uploaded => Create Table with UI => Preview table (cluster should be created and running) => Make sure that the first row is header selected => Create Table. Table will be created with name that you specified (sales_december_2019_csv by default) You will be able to change the table name in the "create table" section. Here are some hints:
  1. DBFS root path - "/FileStore" and all your loaded .csv files are stored in the "/FileStore/tables" folder
  2. You can check your .csv files in the DBFS by the cell below

In [0]:
# dbutils - special utility for databricks that helps to work with a notebook. With this utility you can execute another notebook by dbutils.notebook.run("<notebook_name>") or add any dynamic parameters using dbutils.widgets. or you can access DBFS using code below

filenames = dbutils.fs.ls("/FileStore/tables")
for filename in filenames:
    print(filename.name)

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-4694853016895052>, line 3
      1 # dbutils - special utility for databricks that helps to work with a notebook. With this utility you can execute another notebook by dbutils.notebook.run("<notebook_name>") or add any dynamic parameters using dbutils.widgets. or you can access DBFS using code below
----> 3 filenames = dbutils.fs.ls("/FileStore/tables")
      4 for filename in filenames:
      5     print(filename.name)

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:52, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     49 class ExecutionError(Exception):
     50     pass
---> 52 raise ExecutionError(str(e)) from None

ExecutionError: [DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /FileStore/tables SQLSTATE: 56038

JVM stacktrace:


PySpark can process SQL queries as a text. In other words you don't need to switch cell language to SQL.
1. Write data from table that you created into the dataframe using PySpark with SQL query. Show data in the dataframe

In [0]:
df = spark.sql("SELECT * FROM workspace.default.sales_december_2019")
display(df)

Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
295665,Macbook Pro Laptop,1,1700,12/30/19 00:01,"136 Church St, New York City, NY 10001"
295666,LG Washing Machine,1,600.0,12/29/19 07:03,"562 2nd St, New York City, NY 10001"
295667,USB-C Charging Cable,1,11.95,12/12/19 18:21,"277 Main St, New York City, NY 10001"
295668,27in FHD Monitor,1,149.99,12/22/19 15:13,"410 6th St, San Francisco, CA 94016"
295669,USB-C Charging Cable,1,11.95,12/18/19 12:38,"43 Hill St, Atlanta, GA 30301"
295670,AA Batteries (4-pack),1,3.84,12/31/19 22:58,"200 Jefferson St, New York City, NY 10001"
295671,USB-C Charging Cable,1,11.95,12/16/19 15:10,"928 12th St, Portland, OR 97035"
295672,USB-C Charging Cable,2,11.95,12/13/19 09:29,"813 Hickory St, Dallas, TX 75001"
295673,Bose SoundSport Headphones,1,99.99,12/15/19 23:26,"718 Wilson St, Dallas, TX 75001"
295674,AAA Batteries (4-pack),4,2.99,12/28/19 11:51,"77 7th St, Dallas, TX 75001"


Any notebook can be parameterized using dbutils.widgets. Try to add one parameter "Product_name" and select data from dataframe filtered by value from this parameter. 

2. Select data where product = "product_name" from dataframe using PySpark

In [0]:
dbutils.widgets.text("Product_name", "Macbook Pro Laptop")
selected_product = dbutils.widgets.get("Product_name")
query = f"SELECT * FROM workspace.default.sales_december_2019 WHERE Product = '{selected_product}'"
df_filtered = spark.sql(query)
display(df_filtered)

Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
295665,Macbook Pro Laptop,1,1700,12/30/19 00:01,"136 Church St, New York City, NY 10001"
295712,Macbook Pro Laptop,1,1700,12/10/19 20:02,"331 Madison St, New York City, NY 10001"
295717,Macbook Pro Laptop,1,1700,12/25/19 09:51,"82 10th St, San Francisco, CA 94016"
295871,Macbook Pro Laptop,1,1700,12/28/19 11:19,"661 Park St, Dallas, TX 75001"
295948,Macbook Pro Laptop,1,1700,12/17/19 21:08,"863 West St, San Francisco, CA 94016"
295963,Macbook Pro Laptop,1,1700,12/08/19 10:21,"556 11th St, Austin, TX 73301"
296030,Macbook Pro Laptop,1,1700,12/24/19 12:31,"698 4th St, Portland, OR 97035"
296068,Macbook Pro Laptop,1,1700,12/08/19 22:10,"897 Jackson St, San Francisco, CA 94016"
296076,Macbook Pro Laptop,1,1700,12/03/19 15:19,"679 Chestnut St, San Francisco, CA 94016"
296126,Macbook Pro Laptop,1,1700,12/31/19 18:45,"15 Main St, Seattle, WA 98101"


As well as in SQL, in PySpark you can use aggregate functions. Package pyspark.sql.functions contains all aggregated function from SQL. Try to perform simple aggregation with dataframe. Don't forget, that column types, which you want to calculate, shoud be numerical.  
3. Calculate the sales for each product, including the number of products sold

In [0]:
from pyspark.sql.functions import col, sum, expr
df_numeric = (
    df
    .withColumn("Quantity Ordered", expr("try_cast(`Quantity Ordered` as int)"))
    .withColumn("Price Each", expr("try_cast(`Price Each` as double)"))
    .filter(col("Quantity Ordered").isNotNull() & col("Price Each").isNotNull()))
df_with_sales = df_numeric.withColumn("Sales", col("Quantity Ordered") * col("Price Each"))
df_result = df_with_sales.groupBy("Product").agg(
    sum("Sales").alias("Total_Sales"),
    sum("Quantity Ordered").alias("Total_Quantity_Sold"))
display(df_result)


Product,Total_Sales,Total_Quantity_Sold
Macbook Pro Laptop,1094800.0,644
LG Washing Machine,48000.0,80
USB-C Charging Cable,38849.45000000006,3251
27in FHD Monitor,144740.3500000011,965
AA Batteries (4-pack),14277.120000000394,3718
Bose SoundSport Headphones,182481.74999999825,1825
AAA Batteries (4-pack),12677.599999999413,4240
ThinkPad Laptop,540994.5899999965,541
Lightning Charging Cable,46180.54999999874,3089
Google Phone,429600.0,716


In the PySpark you can perform dataframe profiling using one of two special commands or simple aggregated functions. Try to find special commands to complete this task or just use aggregated functions. Hint: please, сhange the column data types based on the data in them

4. Show data profiles output for the new dataframe of table sales_december_2019_csv: row count, min and max value for each column

In [0]:
df.printSchema()
from pyspark.sql.functions import expr

df_typed = (
    df
    .withColumn("Order ID", expr("try_cast(`Order ID` as int)"))
    .withColumn("Quantity Ordered", expr("try_cast(`Quantity Ordered` as int)"))
    .withColumn("Price Each", expr("try_cast(`Price Each` as double)"))
    .withColumn("Order Date", expr("try_to_timestamp(`Order Date`, 'MM/dd/yy HH:mm')")))
dbutils.data.summarize(df_typed)


root
 |-- Order ID: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity Ordered: string (nullable = true)
 |-- Price Each: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Purchase Address: string (nullable = true)



<!DOCTYPE html>


5. Add new column to the dataframe from previous task with any default value that you want

In [0]:
from pyspark.sql.functions import lit,when
df_with_discount = df_typed.withColumn(
    "Discount", 
    when(col("Quantity Ordered") >= 3, lit(0.15)) 
    .when(col("Quantity Ordered") == 2, lit(0.05)) 
    .otherwise(lit(0.0)))
display(df_with_discount.limit(10))


Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,Discount
295665,Macbook Pro Laptop,1,1700.0,2019-12-30T00:01:00.000Z,"136 Church St, New York City, NY 10001",0.0
295666,LG Washing Machine,1,600.0,2019-12-29T07:03:00.000Z,"562 2nd St, New York City, NY 10001",0.0
295667,USB-C Charging Cable,1,11.95,2019-12-12T18:21:00.000Z,"277 Main St, New York City, NY 10001",0.0
295668,27in FHD Monitor,1,149.99,2019-12-22T15:13:00.000Z,"410 6th St, San Francisco, CA 94016",0.0
295669,USB-C Charging Cable,1,11.95,2019-12-18T12:38:00.000Z,"43 Hill St, Atlanta, GA 30301",0.0
295670,AA Batteries (4-pack),1,3.84,2019-12-31T22:58:00.000Z,"200 Jefferson St, New York City, NY 10001",0.0
295671,USB-C Charging Cable,1,11.95,2019-12-16T15:10:00.000Z,"928 12th St, Portland, OR 97035",0.0
295672,USB-C Charging Cable,2,11.95,2019-12-13T09:29:00.000Z,"813 Hickory St, Dallas, TX 75001",0.05
295673,Bose SoundSport Headphones,1,99.99,2019-12-15T23:26:00.000Z,"718 Wilson St, Dallas, TX 75001",0.0
295674,AAA Batteries (4-pack),4,2.99,2019-12-28T11:51:00.000Z,"77 7th St, Dallas, TX 75001",0.15


Temporary views are processed by cluster and always dropped when the session ends (when the cluster turns off).

6. Create temporary view from task 4 dataframe using PySpark and perform any select using SQL

In [0]:
df_with_discount.createOrReplaceTempView("v_sales_with_discount")


In [0]:
%sql
SELECT Product, `Quantity Ordered`, `Price Each`, Discount
FROM v_sales_with_discount
WHERE Discount > 0
LIMIT 10;


Product,Quantity Ordered,Price Each,Discount
USB-C Charging Cable,2,11.95,0.05
AAA Batteries (4-pack),4,2.99,0.15
USB-C Charging Cable,2,11.95,0.05
AA Batteries (4-pack),2,3.84,0.05
AAA Batteries (4-pack),2,2.99,0.05
AAA Batteries (4-pack),4,2.99,0.15
AA Batteries (4-pack),2,3.84,0.05
USB-C Charging Cable,2,11.95,0.05
Wired Headphones,2,11.99,0.05
AAA Batteries (4-pack),3,2.99,0.15


When we export dataframe to the any file, databricks creates folder with files which are divided into separate files of the same size.

7. Export dataframe from task 5 as .parquet file to the DBFS and show that databricks creates folder with .parquet files

In [0]:
df_with_discount.write \
    .mode("overwrite") \
    .format("parquet") \
    .save("/Volumes/workspace/default/my_files/sales_parquet_folder")
display(dbutils.fs.ls("/Volumes/workspace/default/my_files/sales_parquet_folder"))



path,name,size,modificationTime
dbfs:/Volumes/workspace/default/my_files/sales_parquet_folder/_SUCCESS,_SUCCESS,0,1781727679000
dbfs:/Volumes/workspace/default/my_files/sales_parquet_folder/_committed_102764049156193901,_committed_102764049156193901,123,1781726523000
dbfs:/Volumes/workspace/default/my_files/sales_parquet_folder/_committed_3148859634496115420,_committed_3148859634496115420,233,1781727679000
dbfs:/Volumes/workspace/default/my_files/sales_parquet_folder/_started_102764049156193901,_started_102764049156193901,0,1781726522000
dbfs:/Volumes/workspace/default/my_files/sales_parquet_folder/_started_3148859634496115420,_started_3148859634496115420,0,1781727678000
dbfs:/Volumes/workspace/default/my_files/sales_parquet_folder/part-00000-tid-3148859634496115420-caeffdab-af07-4ea1-a932-cf8e45cee302-374-1.c000.snappy.parquet,part-00000-tid-3148859634496115420-caeffdab-af07-4ea1-a932-cf8e45cee302-374-1.c000.snappy.parquet,589022,1781727678000


root
 |-- Order ID: integer (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity Ordered: integer (nullable = true)
 |-- Price Each: double (nullable = true)
 |-- Order Date: timestamp (nullable = true)
 |-- Purchase Address: string (nullable = true)
 |-- Discount: double (nullable = true)



As well as export, you can import dataframe from files using both SQL and PySpark.

8. Create table from .parquet files that was created in the previous cell using pure SQL  

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.sales_imported_from_parquet
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
AS 
SELECT * FROM parquet.`/Volumes/workspace/default/my_files/sales_parquet_folder`;
SELECT * FROM workspace.default.sales_imported_from_parquet LIMIT 10;

Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,Discount
295665,Macbook Pro Laptop,1,1700.0,2019-12-30T00:01:00.000Z,"136 Church St, New York City, NY 10001",0.0
295666,LG Washing Machine,1,600.0,2019-12-29T07:03:00.000Z,"562 2nd St, New York City, NY 10001",0.0
295667,USB-C Charging Cable,1,11.95,2019-12-12T18:21:00.000Z,"277 Main St, New York City, NY 10001",0.0
295668,27in FHD Monitor,1,149.99,2019-12-22T15:13:00.000Z,"410 6th St, San Francisco, CA 94016",0.0
295669,USB-C Charging Cable,1,11.95,2019-12-18T12:38:00.000Z,"43 Hill St, Atlanta, GA 30301",0.0
295670,AA Batteries (4-pack),1,3.84,2019-12-31T22:58:00.000Z,"200 Jefferson St, New York City, NY 10001",0.0
295671,USB-C Charging Cable,1,11.95,2019-12-16T15:10:00.000Z,"928 12th St, Portland, OR 97035",0.0
295672,USB-C Charging Cable,2,11.95,2019-12-13T09:29:00.000Z,"813 Hickory St, Dallas, TX 75001",0.05
295673,Bose SoundSport Headphones,1,99.99,2019-12-15T23:26:00.000Z,"718 Wilson St, Dallas, TX 75001",0.0
295674,AAA Batteries (4-pack),4,2.99,2019-12-28T11:51:00.000Z,"77 7th St, Dallas, TX 75001",0.15


9. Save previous dataframe as one .csv file into DBFS and show this file using dbutils 

In [0]:
df_typed.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/my_files/sales_december_csv_export")

df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/workspace/default/my_files/sales_december_csv_export")
display(df_csv.limit(10))

Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
295665,Macbook Pro Laptop,1,1700.0,2019-12-30T00:01:00.000Z,"136 Church St, New York City, NY 10001"
295666,LG Washing Machine,1,600.0,2019-12-29T07:03:00.000Z,"562 2nd St, New York City, NY 10001"
295667,USB-C Charging Cable,1,11.95,2019-12-12T18:21:00.000Z,"277 Main St, New York City, NY 10001"
295668,27in FHD Monitor,1,149.99,2019-12-22T15:13:00.000Z,"410 6th St, San Francisco, CA 94016"
295669,USB-C Charging Cable,1,11.95,2019-12-18T12:38:00.000Z,"43 Hill St, Atlanta, GA 30301"
295670,AA Batteries (4-pack),1,3.84,2019-12-31T22:58:00.000Z,"200 Jefferson St, New York City, NY 10001"
295671,USB-C Charging Cable,1,11.95,2019-12-16T15:10:00.000Z,"928 12th St, Portland, OR 97035"
295672,USB-C Charging Cable,2,11.95,2019-12-13T09:29:00.000Z,"813 Hickory St, Dallas, TX 75001"
295673,Bose SoundSport Headphones,1,99.99,2019-12-15T23:26:00.000Z,"718 Wilson St, Dallas, TX 75001"
295674,AAA Batteries (4-pack),4,2.99,2019-12-28T11:51:00.000Z,"77 7th St, Dallas, TX 75001"


9.1 Optional check - create temporary view from .csv file using Spark SQL as you did in task 8

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW sales_csv_view
USING csv
OPTIONS (
  path "/Volumes/workspace/default/my_files/sales_december_csv_export",
  header "true",
  inferSchema "true"
);
SELECT * FROM sales_csv_view LIMIT 10;


Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
295665,Macbook Pro Laptop,1,1700.0,2019-12-30T00:01:00.000Z,"136 Church St, New York City, NY 10001"
295666,LG Washing Machine,1,600.0,2019-12-29T07:03:00.000Z,"562 2nd St, New York City, NY 10001"
295667,USB-C Charging Cable,1,11.95,2019-12-12T18:21:00.000Z,"277 Main St, New York City, NY 10001"
295668,27in FHD Monitor,1,149.99,2019-12-22T15:13:00.000Z,"410 6th St, San Francisco, CA 94016"
295669,USB-C Charging Cable,1,11.95,2019-12-18T12:38:00.000Z,"43 Hill St, Atlanta, GA 30301"
295670,AA Batteries (4-pack),1,3.84,2019-12-31T22:58:00.000Z,"200 Jefferson St, New York City, NY 10001"
295671,USB-C Charging Cable,1,11.95,2019-12-16T15:10:00.000Z,"928 12th St, Portland, OR 97035"
295672,USB-C Charging Cable,2,11.95,2019-12-13T09:29:00.000Z,"813 Hickory St, Dallas, TX 75001"
295673,Bose SoundSport Headphones,1,99.99,2019-12-15T23:26:00.000Z,"718 Wilson St, Dallas, TX 75001"
295674,AAA Batteries (4-pack),4,2.99,2019-12-28T11:51:00.000Z,"77 7th St, Dallas, TX 75001"
